In [ ]:
# === 노트북 공통 preamble (llm_math 패키지 로드) ===
import sys
from pathlib import Path

# llm_math 패키지를 찾을 수 있는 후보 경로들
_candidates = [
    '.', 'src', '..', '../src',
    '/content/llm-math-book/src',
    '/content/llm-math-book',
    '/workspace/src',
    '/workspace',
]
# 현재 디렉토리의 상위 디렉토리들도 후보에 추가 (notebooks/ 폴더에서 실행 시)
for p in Path.cwd().parents:
    _candidates.append(str(p / 'src'))
    _candidates.append(str(p))

for p in _candidates:
    if p and p not in sys.path and Path(p).exists():
        sys.path.insert(0, p)

# llm_math import 시도
try:
    from llm_math import viz, bench, data
    _LLM_MATH_OK = True
except ImportError as e:
    _LLM_MATH_OK = False
    print(f"[주의] llm_math 패키지 로드 실패: {e}")
    print("  GitHub 레포를 클론하고 colab_setup.sh를 실행하세요.")
# === preamble 끝 ===


# Ch 20. 지도 파인튜닝 (SFT) — 지시 따르기

> **학습 목표**
> - SFT가 사전학습 모델을 어떻게 "지시 따르는" 모델로 바꾸는지 이해한다
> - Instruction-response 데이터 포맷을 설계한다
> - 손실 마스킹(loss masking)의 역할을 안다

## 20.1 사전학습 → SFT → RLHF 파이프라인

1. **사전학습**: 다음 토큰 예측 (수 TB 데이터)
2. **SFT**: instruction-response 쌍으로 지시 따르기 학습 (수만~수백만 샘플)
3. **RLHF/DPO**: 인간 선호로 정렬 (수만 샘플)

SFT는 "질문에 답하는" 행동을 가르침. 사전학습 모델은 그냥 텍스트 continuation만 함.

## 20.2 Instruction-Response 데이터 형식

예시 (Alpaca 형식):
```
Below is an instruction... ### Instruction: {instruction} ### Response: {response}
```

ChatML (최신):
```
<|im_start|>user\n{user_msg}<|im_end|>\n<|im_start|>assistant\n{assistant_msg}<|im_end|>
```

## 20.3 손실 마스킹 — 프롬프트 부분은 학습 안 함

$$\mathcal{L}_{\text{SFT}} = -\frac{1}{|\mathbf{y}|} \sum_{t=1}^{|\mathbf{y}|} \log P(y_t | \mathbf{x}, \mathbf{y}_{<t}; \theta)$$

**핵심**: 프롬프트 $\mathbf{x}$ 부분의 토큰은 손실 계산에서 제외. 응답 $\mathbf{y}$ 부분만 학습.
- 이유: 프롬프트를 "생성"하는 법은 원치 않음. 응답을 잘 생성하는 법을 배워야 함.


In [ ]:

# 공통 모델 정의 (Ch 18과 동일)
import torch
import torch.nn as nn
import torch.nn.functional as F

class TransformerBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout=0.1):
        super().__init__()
        self.n_heads = n_heads
        self.d_k = d_model // n_heads
        self.W_qkv = nn.Linear(d_model, 3 * d_model, bias=False)
        self.W_o = nn.Linear(d_model, d_model, bias=False)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_ff), nn.GELU(), nn.Linear(d_ff, d_model)
        )
        self.ln1 = nn.LayerNorm(d_model)
        self.ln2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def attention(self, x, mask):
        B, T, D = x.shape
        q, k, v = self.W_qkv(x).chunk(3, dim=-1)
        q = q.view(B, T, self.n_heads, self.d_k).transpose(1, 2)
        k = k.view(B, T, self.n_heads, self.d_k).transpose(1, 2)
        v = v.view(B, T, self.n_heads, self.d_k).transpose(1, 2)
        scores = q @ k.transpose(-1, -2) / (self.d_k ** 0.5) + mask
        weights = F.softmax(scores, dim=-1)
        out = (weights @ v).transpose(1, 2).contiguous().view(B, T, D)
        return self.W_o(out)

    def forward(self, x, mask):
        x = x + self.dropout(self.attention(self.ln1(x), mask))
        x = x + self.dropout(self.ffn(self.ln2(x)))
        return x

class MiniGPT(nn.Module):
    def __init__(self, vocab_size, d_model=64, n_heads=4, n_layers=2,
                 d_ff=256, max_seq_len=128, dropout=0.1):
        super().__init__()
        self.d_model = d_model
        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(max_seq_len, d_model)
        self.blocks = nn.ModuleList([
            TransformerBlock(d_model, n_heads, d_ff, dropout)
            for _ in range(n_layers)
        ])
        self.ln_f = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        self.lm_head.weight = self.token_emb.weight  # weight tying
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T = x.shape
        positions = torch.arange(T, device=x.device).unsqueeze(0)
        emb = self.token_emb(x) * (self.d_model ** 0.5) + self.pos_emb(positions)
        h = self.dropout(emb)
        mask = torch.triu(torch.full((T, T), float('-inf'), device=x.device), diagonal=1)
        for block in self.blocks:
            h = block(h, mask)
        h = self.ln_f(h)
        return self.lm_head(h)

print("MiniGPT Model 정의 완료")


In [ ]:
# 미니 SFT 데이터셋
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

# 간단한 instruction-response 예시 (실제로는 더 큰 데이터)
sft_examples = [
    {"instruction": "What is 2+2?", "response": "2+2 equals 4."},
    {"instruction": "Translate hello to Korean.", "response": "hello in Korean is 안녕하세요."},
    {"instruction": "Capital of France?", "response": "The capital of France is Paris."},
    {"instruction": "Define AI.", "response": "AI is the simulation of human intelligence in machines."},
    {"instruction": "Reverse abc.", "response": "cba"},
]

# 간단한 Character-level Tokenizer
all_text = " ".join([f"{e['instruction']} {e['response']}" for e in sft_examples])
chars = sorted(set(all_text))
vocab_size = len(chars)
char_to_idx = {c: i for i, c in enumerate(chars)}
idx_to_char = {i: c for i, c in enumerate(chars)}
print(f"Vocabulary Size: {vocab_size}")

def encode(text):
    return [char_to_idx[c] for c in text if c in char_to_idx]

def format_example(ex):
    """Alpaca 스타일 포맷."""
    return f"### Instruction: {ex['instruction']}### Response: {ex['response']}<END>"

# Loss 마스킹 포함 Batch
def make_sft_batch(examples, max_len=128):
    """instruction은 0, response는 1로 Mask."""
    inputs, masks = [], []
    for ex in examples:
        formatted = format_example(ex)
        ids = encode(formatted)[:max_len]
        # Response Subspace 식별
        resp_start = formatted.find("### Response:") + len("### Response:")
        full_text = formatted
        # mask: 0 for instruction, 1 for response
        mask = [0] * len(formatted)
        for i in range(resp_start, len(formatted)):
            mask[i] = 1
        # 자르기
        mask = mask[:len(ids)]
        # 패딩
        while len(ids) < max_len:
            ids.append(0)
            mask.append(0)
        inputs.append(ids)
        masks.append(mask)
    return (torch.tensor(inputs, dtype=torch.long),
            torch.tensor(masks, dtype=torch.float32))

x, mask = make_sft_batch(sft_examples, max_len=128)
print(f"입력: {x.shape}, 마스크: {mask.shape}")
print(f"Mean Response 토큰 Ratio: {mask.mean():.4f}")


In [ ]:
# SFT 학습 루프
def sft_loss_with_mask(logits, targets, mask):
    """마스킹을 Application한 교차 Entropy Loss."""
    # logits: (B, T, V), targets: (B, T), mask: (B, T)
    B, T, V = logits.shape
    # shift: Input [0..T-2], Label [1..T-1]
    logits = logits[:, :-1, :].reshape(-1, V)
    targets = targets[:, 1:].reshape(-1)
    mask = mask[:, 1:].reshape(-1)
    # token별 Loss
    losses = F.cross_entropy(logits, targets, reduction='none')
    # Mask Application
    masked_loss = (losses * mask).sum() / (mask.sum() + 1e-8)
    return masked_loss

# Training
model = MiniGPT(vocab_size, d_model=32, n_heads=4, n_layers=2, d_ff=128, max_seq_len=128)
opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=0.01)

# Loss 마스킹 ON vs OFF Comparison
import matplotlib.pyplot as plt

# 200 Step Training (마스킹 O)
torch.manual_seed(0)
model_with_mask = MiniGPT(vocab_size, d_model=32, n_heads=4, n_layers=2, d_ff=128, max_seq_len=128)
opt1 = torch.optim.AdamW(model_with_mask.parameters(), lr=1e-3)
losses_masked = []
for step in range(200):
    x, mask = make_sft_batch(sft_examples, max_len=128)
    opt1.zero_grad()
    logits = model_with_mask(x)
    loss = sft_loss_with_mask(logits, x, mask)
    loss.backward()
    opt1.step()
    losses_masked.append(loss.item())

plt.figure(figsize=(10, 4))
plt.plot(losses_masked, label='SFT with loss masking', color='blue')
plt.xlabel('Step'); plt.ylabel('Loss')
plt.title('SFT Learning Curve (Response Subspace만 Training)')
plt.legend(); plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../figures/ch20_sft.png', dpi=100, bbox_inches='tight')
plt.show()
print(f"최종 손실: {losses_masked[-1]:.4f}")


In [ ]:
# SFT 후 응답 생성
def generate_sft(model, instruction, max_new=80, temperature=0.7):
    model.eval()
    prompt = f"### Instruction: {instruction}### Response:"
    idx = encode(prompt)
    with torch.no_grad():
        for _ in range(max_new):
            x = torch.tensor([idx[-128:]], dtype=torch.long)
            logits = model(x)
            logit = logits[0, -1] / temperature
            probs = F.softmax(logit, dim=0)
            next_idx = torch.multinomial(probs, 1).item()
            if next_idx == char_to_idx.get('<', -1):
                break
            idx.append(next_idx)
            # <END>에서 멈춤 (간단히)
            text_so_far = ''.join(idx_to_char.get(i, '') for i in idx)
            if '<END>' in text_so_far:
                break
    return ''.join(idx_to_char.get(i, '') for i in idx)

print("=== SFT 후 응답 생성 ===\n")
for inst in ["What is 2+2?", "Capital of France?", "Reverse abc."]:
    print(f"Q: {inst}")
    print(f"A: {generate_sft(model_with_mask, inst, max_new=80, temperature=0.5)}")
    print()


## 20.4 데이터 품질이 성능을 좌우한다 — LIMA의 교훈

LIMA (Meta, 2023) 실험:
- 사전학습 모델 + 단 1,000개의 고품질 instruction-response로 SFT
- 65B LLaMA로 GPT-4 수준 응답 가능

교훈: **데이터 양보다 품질**. 사전학습에서 이미 많은 것을 배웠다. SFT는 "표현 형식"을 가르치는 정도면 충분.

## 20.5 전체 파인튜닝 vs PEFT (Parameter-Efficient Fine-Tuning)

- **전체 파인튜닝**: 모든 파라미터 갱신. 비용 큼.
- **PEFT**: 일부 파라미터만 갱신 (LoRA, Adapter, Prefix Tuning 등). Ch 26에서 상세.

## 20.6 핵심 정리

| 개념 | 설명 |
|---|---|
| SFT | instruction-response로 지시 따르기 학습 |
| 손실 마스킹 | 프롬프트 부분은 손실 제외 |
| 학습률 | 사전학습의 1/10 수준 |
| 데이터 품질 | 양보다 질 (LIMA 교훈) |

## 연습문제

1. 손실 마스킹을 켜고 끄고 각각 학습 후 응답 품질을 비교하라.
2. 학습률 1e-2, 1e-3, 1e-4로 SFT하고 비교하라.
3. instruction 데이터 5, 50, 500개로 늘려가며 SFT 효과를 비교하라.
4. ChatML 포맷으로 SFT 데이터를 변환하라.
5. 사전학습 모델 vs SFT 모델의 응답을 비교하고 차이를 설명하라.

> 해설: `solutions/ch20_solutions.ipynb`
